In [1]:
import torch

if torch.cuda.is_available():
    print("GPU is available:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")


GPU is available: Tesla T4


In [2]:
  !pip install ptflops


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
from torchvision.models import resnet18, resnet50
import time
import matplotlib.pyplot as plt


In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

mnist = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
fashion = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)

def split_dataset(dataset):
    total = len(dataset)
    train_size = int(0.7 * total)
    val_size = int(0.1 * total)
    test_size = total - train_size - val_size
    return random_split(dataset, [train_size, val_size, test_size])

mnist_train, mnist_val, mnist_test = split_dataset(mnist)
fashion_train, fashion_val, fashion_test = split_dataset(fashion)

print(len(mnist_train), len(mnist_val), len(mnist_test))


100%|██████████| 9.91M/9.91M [00:00<00:00, 18.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 484kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.76MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.5MB/s]
100%|██████████| 26.4M/26.4M [00:02<00:00, 9.90MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 217kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.42MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 27.1MB/s]

42000 6000 12000


In [5]:
from torchvision.models import resnet18, resnet50
import torch.nn as nn

def get_resnet(model_name="resnet18"):
    if model_name == "resnet18":
        model = resnet18(pretrained=False)
    elif model_name == "resnet50":
        model = resnet50(pretrained=False)
    else:
        raise ValueError("Only resnet18 and resnet50 supported")

    # Change first layer for 1-channel images
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

    # Change last layer for 10 classes
    model.fc = nn.Linear(model.fc.in_features, 10)

    return model


In [6]:
from torch.utils.data import DataLoader

batch_size = 16

mnist_train_loader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True, pin_memory=True)
mnist_val_loader   = DataLoader(mnist_val, batch_size=batch_size, shuffle=False, pin_memory=True)
mnist_test_loader  = DataLoader(mnist_test, batch_size=batch_size, shuffle=False, pin_memory=True)

fashion_train_loader = DataLoader(fashion_train, batch_size=batch_size, shuffle=True, pin_memory=True)
fashion_val_loader   = DataLoader(fashion_val, batch_size=batch_size, shuffle=False, pin_memory=True)
fashion_test_loader  = DataLoader(fashion_test, batch_size=batch_size, shuffle=False, pin_memory=True)


In [7]:
import torch
import time

def train_model(model, train_loader, val_loader, optimizer, epochs, device):
    criterion = nn.CrossEntropyLoss()
    model.to(device)

    train_acc_list = []
    val_acc_list = []

    for epoch in range(epochs):
        model.train()
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc = 100 * correct / total
        train_acc_list.append(train_acc)

        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = 100 * correct / total
        val_acc_list.append(val_acc)

        print(f"Epoch [{epoch+1}/{epochs}] → Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

    return train_acc_list, val_acc_list


In [8]:
def test_model(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_acc = 100 * correct / total
    print(f"Final Test Accuracy: {test_acc:.2f}%")
    return test_acc


## Experiment 1  
Dataset: MNIST  
Model: ResNet-18  
Batch Size: 16  
Optimizer: SGD  
Learning Rate: 0.001  
Epochs: 5  


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = get_resnet("resnet18")
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

train_acc, val_acc = train_model(
    model,
    mnist_train_loader,
    mnist_val_loader,
    optimizer,
    epochs=5,   # First experiment: 5 epochs
    device=device
)

test_accuracy = test_model(model, mnist_test_loader, device)


Epoch [1/5] → Train Acc: 89.49% | Val Acc: 96.98%
Epoch [2/5] → Train Acc: 96.25% | Val Acc: 97.70%
Epoch [3/5] → Train Acc: 97.64% | Val Acc: 98.07%
Epoch [4/5] → Train Acc: 98.25% | Val Acc: 98.15%
Epoch [5/5] → Train Acc: 98.54% | Val Acc: 98.28%
Final Test Accuracy: 98.37%


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(train_acc, label="Train Accuracy")
plt.plot(val_acc, label="Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("MNIST - ResNet18 - Batch16 - SGD - LR 0.001")
plt.legend()
plt.show()


## Experiment 2  
Dataset: MNIST  
Model: ResNet-18  
Batch Size: 16  
Optimizer: SGD  
Learning Rate: 0.0001  
Epochs: 5  

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = get_resnet("resnet18")
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)

train_acc, val_acc = train_model(
    model,
    mnist_train_loader,
    mnist_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, mnist_test_loader, device)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch [1/5] → Train Acc: 65.07% | Val Acc: 86.83%
Epoch [2/5] → Train Acc: 87.15% | Val Acc: 91.65%
Epoch [3/5] → Train Acc: 90.42% | Val Acc: 93.62%
Epoch [4/5] → Train Acc: 92.02% | Val Acc: 94.37%
Epoch [5/5] → Train Acc: 93.23% | Val Acc: 94.97%
Final Test Accuracy: 95.07%


## Experiment 3  
Dataset: MNIST  
Model: ResNet-18  
Batch Size: 16  
Optimizer: Adam  
Learning Rate: 0.001  
Epochs: 5  

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = get_resnet("resnet18")
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

train_acc, val_acc = train_model(
    model,
    mnist_train_loader,
    mnist_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, mnist_test_loader, device)


Epoch [1/5] → Train Acc: 94.86% | Val Acc: 97.05%
Epoch [2/5] → Train Acc: 97.66% | Val Acc: 97.83%
Epoch [3/5] → Train Acc: 98.23% | Val Acc: 98.77%
Epoch [4/5] → Train Acc: 98.53% | Val Acc: 98.55%
Epoch [5/5] → Train Acc: 98.74% | Val Acc: 98.30%
Final Test Accuracy: 98.18%


## Experiment 4
Dataset: MNIST  
Model: ResNet-18  
Batch Size: 16  
Optimizer: Adam  
Learning Rate: 0.0001  
Epochs: 5  


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = get_resnet("resnet18")
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

train_acc, val_acc = train_model(
    model,
    mnist_train_loader,
    mnist_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, mnist_test_loader, device)


Epoch [1/5] → Train Acc: 93.86% | Val Acc: 97.77%
Epoch [2/5] → Train Acc: 97.82% | Val Acc: 97.93%
Epoch [3/5] → Train Acc: 98.48% | Val Acc: 98.73%
Epoch [4/5] → Train Acc: 98.93% | Val Acc: 98.15%
Epoch [5/5] → Train Acc: 98.98% | Val Acc: 98.68%
Final Test Accuracy: 98.65%


In [9]:
from torch.utils.data import DataLoader

batch_size = 32

mnist_train_loader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True, pin_memory=True)
mnist_val_loader   = DataLoader(mnist_val, batch_size=batch_size, shuffle=False, pin_memory=True)
mnist_test_loader  = DataLoader(mnist_test, batch_size=batch_size, shuffle=False, pin_memory=True)

fashion_train_loader = DataLoader(fashion_train, batch_size=batch_size, shuffle=True, pin_memory=True)
fashion_val_loader   = DataLoader(fashion_val, batch_size=batch_size, shuffle=False, pin_memory=True)
fashion_test_loader  = DataLoader(fashion_test, batch_size=batch_size, shuffle=False, pin_memory=True)


## Experiment 5
Dataset: MNIST  
Model: ResNet-18  
Batch Size: 32  
Optimizer: SGD  
Learning Rate: 0.001  
Epochs: 5  


In [ ]:
model = get_resnet("resnet18")
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

train_acc, val_acc = train_model(
    model,
    mnist_train_loader,
    mnist_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, mnist_test_loader, device)


Epoch [1/5] → Train Acc: 86.45% | Val Acc: 94.70%
Epoch [2/5] → Train Acc: 95.60% | Val Acc: 96.27%
Epoch [3/5] → Train Acc: 96.89% | Val Acc: 97.22%
Epoch [4/5] → Train Acc: 97.83% | Val Acc: 97.37%
Epoch [5/5] → Train Acc: 98.37% | Val Acc: 97.62%
Final Test Accuracy: 97.53%


## Experiment 6
Dataset: MNIST  
Model: ResNet-18  
Batch Size: 32  
Optimizer: SGD  
Learning Rate: 0.0001  
Epochs: 5  


In [ ]:
model = get_resnet("resnet18")
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)

train_acc, val_acc = train_model(
    model,
    mnist_train_loader,
    mnist_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, mnist_test_loader, device)

Epoch [1/5] → Train Acc: 53.81% | Val Acc: 77.50%
Epoch [2/5] → Train Acc: 82.30% | Val Acc: 87.20%
Epoch [3/5] → Train Acc: 87.87% | Val Acc: 90.43%
Epoch [4/5] → Train Acc: 90.36% | Val Acc: 92.07%
Epoch [5/5] → Train Acc: 91.72% | Val Acc: 93.20%
Final Test Accuracy: 92.99%


## Experiment 7
Dataset: MNIST  
Model: ResNet-18  
Batch Size: 32  
Optimizer: Adam  
Learning Rate: 0.0001  
Epochs: 5  


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = get_resnet("resnet18")
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

train_acc, val_acc = train_model(
    model,
    mnist_train_loader,
    mnist_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, mnist_test_loader, device)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch [1/5] → Train Acc: 93.99% | Val Acc: 97.92%
Epoch [2/5] → Train Acc: 97.22% | Val Acc: 98.45%
Epoch [3/5] → Train Acc: 97.83% | Val Acc: 98.27%
Epoch [4/5] → Train Acc: 98.36% | Val Acc: 98.35%
Epoch [5/5] → Train Acc: 98.73% | Val Acc: 98.67%
Final Test Accuracy: 98.91%


## Experiment 8
Dataset: MNIST  
Model: ResNet-18  
Batch Size: 32  
Optimizer: Adam  
Learning Rate: 0.0001  
Epochs: 5  


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = get_resnet("resnet18")
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

train_acc, val_acc = train_model(
    model,
    mnist_train_loader,
    mnist_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, mnist_test_loader, device)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch [1/5] → Train Acc: 93.25% | Val Acc: 97.65%
Epoch [2/5] → Train Acc: 97.39% | Val Acc: 98.10%
Epoch [3/5] → Train Acc: 98.22% | Val Acc: 98.08%
Epoch [4/5] → Train Acc: 98.61% | Val Acc: 98.28%
Epoch [5/5] → Train Acc: 98.98% | Val Acc: 98.48%
Final Test Accuracy: 98.69%


## Experiment 9
Dataset: MNIST  
Model: ResNet-50  
Batch Size: 16
Optimizer: SGD  
Learning Rate: 0.001  
Epochs: 5  


In [ ]:


model = get_resnet("resnet50")
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

train_acc, val_acc = train_model(
    model,
    mnist_train_loader,
    mnist_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, mnist_test_loader, device)


Epoch [1/5] → Train Acc: 71.22% | Val Acc: 92.40%
Epoch [2/5] → Train Acc: 91.60% | Val Acc: 95.13%
Epoch [3/5] → Train Acc: 94.23% | Val Acc: 96.80%
Epoch [4/5] → Train Acc: 95.83% | Val Acc: 97.02%
Epoch [5/5] → Train Acc: 96.81% | Val Acc: 97.77%
Final Test Accuracy: 97.94%


## Experiment 10
Dataset: MNIST.  
Model: ResNet-50.   
Batch Size: 16.   
Optimizer: SGD.    
Learning Rate: 0.0001.  
Epochs: 5

In [ ]:
model = get_resnet("resnet50")
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)

train_acc, val_acc = train_model(
    model,
    mnist_train_loader,
    mnist_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, mnist_test_loader, device)

Epoch [1/5] → Train Acc: 18.47% | Val Acc: 29.17%
Epoch [2/5] → Train Acc: 38.81% | Val Acc: 53.57%
Epoch [3/5] → Train Acc: 60.96% | Val Acc: 73.32%
Epoch [4/5] → Train Acc: 74.97% | Val Acc: 82.57%
Epoch [5/5] → Train Acc: 81.44% | Val Acc: 86.60%
Final Test Accuracy: 86.62%


## Experiment 11
Dataset: MNIST  
Model: ResNet-50  
Batch Size: 16
Optimizer: Adam  
Learning Rate: 0.001  
Epochs: 5  

In [ ]:
model = get_resnet("resnet50")
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

train_acc, val_acc = train_model(
    model,
    mnist_train_loader,
    mnist_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, mnist_test_loader, device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch [1/5] → Train Acc: 74.31% | Val Acc: 90.25%
Epoch [2/5] → Train Acc: 92.59% | Val Acc: 95.48%
Epoch [3/5] → Train Acc: 95.66% | Val Acc: 96.52%
Epoch [4/5] → Train Acc: 97.04% | Val Acc: 97.77%
Epoch [5/5] → Train Acc: 97.78% | Val Acc: 97.63%
Final Test Accuracy: 97.94%


## Experiment 12
Dataset: MNIST  
Model: ResNet-50  
Batch Size: 16
Optimizer: Adam  
Learning Rate: 0.0001  
Epochs: 5  

In [ ]:
model = get_resnet("resnet50")
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

train_acc, val_acc = train_model(
    model,
    mnist_train_loader,
    mnist_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, mnist_test_loader, device)

Epoch [1/5] → Train Acc: 72.53% | Val Acc: 89.33%
Epoch [2/5] → Train Acc: 91.76% | Val Acc: 93.87%
Epoch [3/5] → Train Acc: 95.39% | Val Acc: 95.13%
Epoch [4/5] → Train Acc: 96.80% | Val Acc: 95.30%
Epoch [5/5] → Train Acc: 97.65% | Val Acc: 96.75%
Final Test Accuracy: 96.72%


In [10]:
from torch.utils.data import DataLoader

batch_size = 32

mnist_train_loader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True, pin_memory=True)
mnist_val_loader   = DataLoader(mnist_val, batch_size=batch_size, shuffle=False, pin_memory=True)
mnist_test_loader  = DataLoader(mnist_test, batch_size=batch_size, shuffle=False, pin_memory=True)

fashion_train_loader = DataLoader(fashion_train, batch_size=batch_size, shuffle=True, pin_memory=True)
fashion_val_loader   = DataLoader(fashion_val, batch_size=batch_size, shuffle=False, pin_memory=True)
fashion_test_loader  = DataLoader(fashion_test, batch_size=batch_size, shuffle=False, pin_memory=True)


## Experiment 13
Dataset: MNIST  
Model: ResNet-50  
Batch Size: 32      
Optimizer: SGD  
Learning Rate: 0.001  
Epochs: 5  

In [ ]:
model = get_resnet("resnet50")
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

train_acc, val_acc = train_model(
    model,
    mnist_train_loader,
    mnist_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, mnist_test_loader, device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch [1/5] → Train Acc: 56.96% | Val Acc: 85.65%
Epoch [2/5] → Train Acc: 88.63% | Val Acc: 92.20%
Epoch [3/5] → Train Acc: 92.83% | Val Acc: 94.42%
Epoch [4/5] → Train Acc: 94.75% | Val Acc: 95.85%
Epoch [5/5] → Train Acc: 95.76% | Val Acc: 96.17%
Final Test Accuracy: 96.55%


## Experiment 14
Dataset: MNIST  
Model: ResNet-50  
Batch Size: 32      
Optimizer: Adam
Learning Rate: 0.0001  
Epochs: 5  

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = get_resnet("resnet50")
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

train_acc, val_acc = train_model(
    model,
    mnist_train_loader,
    mnist_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, mnist_test_loader, device )

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch [1/5] → Train Acc: 72.20% | Val Acc: 89.12%
Epoch [2/5] → Train Acc: 91.69% | Val Acc: 93.75%
Epoch [3/5] → Train Acc: 95.49% | Val Acc: 95.52%
Epoch [4/5] → Train Acc: 96.87% | Val Acc: 96.27%
Epoch [5/5] → Train Acc: 97.53% | Val Acc: 96.75%
Final Test Accuracy: 96.78%


In [11]:
from torch.utils.data import DataLoader

batch_size = 16  # start again from 16

fashion_train_loader = DataLoader(fashion_train, batch_size=batch_size, shuffle=True, pin_memory=True)
fashion_val_loader   = DataLoader(fashion_val, batch_size=batch_size, shuffle=False, pin_memory=True)
fashion_test_loader  = DataLoader(fashion_test, batch_size=batch_size, shuffle=False, pin_memory=True)


## Experiment 15:
Data: FashionMNIST    
Model:ResNet-18      
Batch Size: 16    
Optimizer: SGD    
LR=0.001     
Epochs=5


In [ ]:
model = get_resnet("resnet18")
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

train_acc, val_acc = train_model(
    model,
    fashion_train_loader,
    fashion_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, fashion_test_loader, device)


Epoch [1/5] → Train Acc: 77.70% | Val Acc: 84.55%
Epoch [2/5] → Train Acc: 85.23% | Val Acc: 86.53%
Epoch [3/5] → Train Acc: 87.27% | Val Acc: 87.37%
Epoch [4/5] → Train Acc: 88.72% | Val Acc: 87.78%
Epoch [5/5] → Train Acc: 90.13% | Val Acc: 87.92%
Final Test Accuracy: 88.18%


## Experiment 16:
Data: FashionMNIST    
Model:ResNet-18      
Batch Size: 16    
Optimizer: SGD    
LR=0.0001     
Epochs=5


In [ ]:
model = get_resnet("resnet18")
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)

train_acc, val_acc = train_model(
    model,
    fashion_train_loader,
    fashion_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, fashion_test_loader, device)


Epoch [1/5] → Train Acc: 59.93% | Val Acc: 74.93%
Epoch [2/5] → Train Acc: 74.77% | Val Acc: 79.23%
Epoch [3/5] → Train Acc: 77.74% | Val Acc: 80.98%
Epoch [4/5] → Train Acc: 79.70% | Val Acc: 82.15%
Epoch [5/5] → Train Acc: 81.00% | Val Acc: 83.20%
Final Test Accuracy: 82.85%


## Experiment 17:
Data: FashionMNIST    
Model:ResNet-18      
Batch Size: 16    
Optimizer: Adam   
LR=0.001     
Epochs=5


In [ ]:
model = get_resnet("resnet18")
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

train_acc, val_acc = train_model(
    model,
    fashion_train_loader,
    fashion_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, fashion_test_loader, device)

Epoch [1/5] → Train Acc: 77.72% | Val Acc: 84.80%
Epoch [2/5] → Train Acc: 85.43% | Val Acc: 86.03%
Epoch [3/5] → Train Acc: 87.26% | Val Acc: 87.18%
Epoch [4/5] → Train Acc: 88.84% | Val Acc: 87.72%
Epoch [5/5] → Train Acc: 90.00% | Val Acc: 87.52%
Final Test Accuracy: 88.35%


## Experiment 18:
Data: FashionMNIST    
Model:ResNet-18      
Batch Size: 16    
Optimizer: Adam   
LR=0.0001     
Epochs=5


In [ ]:
model = get_resnet("resnet18")
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)

train_acc, val_acc = train_model(
    model,
    fashion_train_loader,
    fashion_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, fashion_test_loader, device)

Epoch [1/5] → Train Acc: 61.44% | Val Acc: 74.25%
Epoch [2/5] → Train Acc: 74.51% | Val Acc: 78.35%
Epoch [3/5] → Train Acc: 77.67% | Val Acc: 80.35%
Epoch [4/5] → Train Acc: 79.64% | Val Acc: 81.43%
Epoch [5/5] → Train Acc: 81.22% | Val Acc: 82.60%
Final Test Accuracy: 82.76%


In [12]:
from torch.utils.data import DataLoader

batch_size = 32

fashion_train_loader = DataLoader(fashion_train, batch_size=batch_size, shuffle=True, pin_memory=True)
fashion_val_loader   = DataLoader(fashion_val, batch_size=batch_size, shuffle=False, pin_memory=True)
fashion_test_loader  = DataLoader(fashion_test, batch_size=batch_size, shuffle=False, pin_memory=True)

print("FashionMNIST loaders updated with batch size =", batch_size)


FashionMNIST loaders updated with batch size = 32


## Experiment 19:
Data: FashionMNIST    
Model:ResNet-18      
Batch Size: 32   
Optimizer: SGD   
LR=0.001     
Epochs=5


In [ ]:
model = get_resnet("resnet18")
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

train_acc, val_acc = train_model(
    model,
    fashion_train_loader,
    fashion_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, fashion_test_loader, device)


Epoch [1/5] → Train Acc: 75.03% | Val Acc: 83.55%
Epoch [2/5] → Train Acc: 84.32% | Val Acc: 85.58%
Epoch [3/5] → Train Acc: 86.88% | Val Acc: 86.10%
Epoch [4/5] → Train Acc: 88.76% | Val Acc: 86.40%
Epoch [5/5] → Train Acc: 90.00% | Val Acc: 86.93%
Final Test Accuracy: 87.22%


## Experiment 20:
Data: FashionMNIST    
Model:ResNet-18      
Batch Size: 32   
Optimizer: Adam  
LR=0.0001     
Epochs=5

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = get_resnet("resnet18")
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

train_acc, val_acc = train_model(
    model,
    fashion_train_loader,
    fashion_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, fashion_test_loader, device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch [1/5] → Train Acc: 82.68% | Val Acc: 86.92%
Epoch [2/5] → Train Acc: 87.52% | Val Acc: 88.82%
Epoch [3/5] → Train Acc: 89.02% | Val Acc: 88.90%
Epoch [4/5] → Train Acc: 90.06% | Val Acc: 89.25%
Epoch [5/5] → Train Acc: 90.95% | Val Acc: 89.97%
Final Test Accuracy: 90.33%


In [13]:
from torch.utils.data import DataLoader

batch_size = 16

fashion_train_loader = DataLoader(fashion_train, batch_size=batch_size, shuffle=True, pin_memory=True)
fashion_val_loader   = DataLoader(fashion_val, batch_size=batch_size, shuffle=False, pin_memory=True)
fashion_test_loader  = DataLoader(fashion_test, batch_size=batch_size, shuffle=False, pin_memory=True)

print("FashionMNIST loaders reset to batch size =", batch_size)


FashionMNIST loaders reset to batch size = 16


## Experiment 21  
Data: FashionMNIST  
Model: ResNet-50  
Batch Size: 16  
Optimizer: SGD  
Learning Rate: 0.001  
Epochs: 5


In [ ]:
model = get_resnet("resnet50")
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

train_acc, val_acc = train_model(
    model,
    fashion_train_loader,
    fashion_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, fashion_test_loader, device)


Epoch [1/5] → Train Acc: 61.44% | Val Acc: 74.70%
Epoch [2/5] → Train Acc: 75.71% | Val Acc: 80.27%
Epoch [3/5] → Train Acc: 79.30% | Val Acc: 82.50%
Epoch [4/5] → Train Acc: 81.60% | Val Acc: 84.02%
Epoch [5/5] → Train Acc: 83.07% | Val Acc: 84.72%
Final Test Accuracy: 84.71%


## Experiment 22  
Data: FashionMNIST  
Model: ResNet-50  
Batch Size: 16  
Optimizer: SGD  
Learning Rate: 0.0001  
Epochs: 5


In [16]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = get_resnet("resnet50")
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)

train_acc, val_acc = train_model(
    model,
    fashion_train_loader,
    fashion_val_loader,
    optimizer,
    epochs=8,
    device=device
)

test_accuracy = test_model(model, fashion_test_loader, device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch [1/8] → Train Acc: 19.60% | Val Acc: 33.43%
Epoch [2/8] → Train Acc: 43.33% | Val Acc: 55.83%
Epoch [3/8] → Train Acc: 58.53% | Val Acc: 66.05%
Epoch [4/8] → Train Acc: 65.06% | Val Acc: 69.58%
Epoch [5/8] → Train Acc: 68.45% | Val Acc: 73.18%
Epoch [6/8] → Train Acc: 70.89% | Val Acc: 74.22%
Epoch [7/8] → Train Acc: 72.21% | Val Acc: 75.38%
Epoch [8/8] → Train Acc: 73.87% | Val Acc: 77.15%
Final Test Accuracy: 77.39%


## Experiment 23  
Data: FashionMNIST  
Model: ResNet-50  
Batch Size: 16  
Optimizer: Adam  
Learning Rate: 0.001  
Epochs: 5


In [17]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = get_resnet("resnet50")
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

train_acc, val_acc = train_model(
    model,
    fashion_train_loader,
    fashion_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, fashion_test_loader, device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch [1/5] → Train Acc: 70.83% | Val Acc: 80.33%
Epoch [2/5] → Train Acc: 79.18% | Val Acc: 72.95%
Epoch [3/5] → Train Acc: 80.33% | Val Acc: 83.53%
Epoch [4/5] → Train Acc: 83.67% | Val Acc: 85.92%
Epoch [5/5] → Train Acc: 84.53% | Val Acc: 86.55%
Final Test Accuracy: 87.18%


## Experiment 24  
Data: FashionMNIST  
Model: ResNet-50  
Batch Size: 16  
Optimizer: Adam  
Learning Rate: 0.0001  
Epochs: 5


In [19]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = get_resnet("resnet50")
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

train_acc, val_acc = train_model(
    model,
    fashion_train_loader,
    fashion_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, fashion_test_loader, device)

Epoch [1/5] → Train Acc: 65.75% | Val Acc: 77.60%
Epoch [2/5] → Train Acc: 79.83% | Val Acc: 83.40%
Epoch [3/5] → Train Acc: 83.90% | Val Acc: 85.30%
Epoch [4/5] → Train Acc: 86.26% | Val Acc: 86.32%
Epoch [5/5] → Train Acc: 88.01% | Val Acc: 87.15%
Final Test Accuracy: 87.92%


In [14]:
from torch.utils.data import DataLoader

batch_size = 32

fashion_train_loader = DataLoader(fashion_train, batch_size=batch_size, shuffle=True, pin_memory=True)
fashion_val_loader   = DataLoader(fashion_val, batch_size=batch_size, shuffle=False, pin_memory=True)
fashion_test_loader  = DataLoader(fashion_test, batch_size=batch_size, shuffle=False, pin_memory=True)

print("FashionMNIST loaders updated with batch size =", batch_size)

FashionMNIST loaders updated with batch size = 32


## Experiment 25  
Data: FashionMNIST  
Model: ResNet-50  
Batch Size: 32  
Optimizer: SGD  
Learning Rate: 0.001  
Epochs: 5


In [21]:
model = get_resnet("resnet50")
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

train_acc, val_acc = train_model(
    model,
    fashion_train_loader,
    fashion_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, fashion_test_loader, device)


Epoch [1/5] → Train Acc: 51.75% | Val Acc: 71.87%
Epoch [2/5] → Train Acc: 74.08% | Val Acc: 78.18%
Epoch [3/5] → Train Acc: 78.06% | Val Acc: 80.83%
Epoch [4/5] → Train Acc: 80.35% | Val Acc: 80.83%
Epoch [5/5] → Train Acc: 81.74% | Val Acc: 83.15%
Final Test Accuracy: 83.83%


## Experiment 26  
Data: FashionMNIST  
Model: ResNet-50  
Batch Size: 32  
Optimizer: Adam  
Learning Rate: 0.0001  
Epochs: 5


In [22]:
model = get_resnet("resnet50")
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

train_acc, val_acc = train_model(
    model,
    fashion_train_loader,
    fashion_val_loader,
    optimizer,
    epochs=5,
    device=device
)

test_accuracy = test_model(model, fashion_test_loader, device)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch [1/5] → Train Acc: 66.55% | Val Acc: 77.48%
Epoch [2/5] → Train Acc: 79.98% | Val Acc: 81.65%
Epoch [3/5] → Train Acc: 83.65% | Val Acc: 83.53%
Epoch [4/5] → Train Acc: 86.13% | Val Acc: 85.48%
Epoch [5/5] → Train Acc: 88.10% | Val Acc: 86.00%
Final Test Accuracy: 86.24%


# Q1(b): SVM Experiments on MNIST and FashionMNIST


In [16]:
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import time

# Function to convert PyTorch dataset to NumPy arrays
def dataset_to_numpy(dataset):
    X = []
    y = []
    for img, label in dataset:
        X.append(img.numpy().flatten())
        y.append(label)
    return np.array(X), np.array(y)

# Convert MNIST
X_mnist_train, y_mnist_train = dataset_to_numpy(mnist_train)
X_mnist_test, y_mnist_test   = dataset_to_numpy(mnist_test)

# Convert FashionMNIST
X_fashion_train, y_fashion_train = dataset_to_numpy(fashion_train)
X_fashion_test, y_fashion_test   = dataset_to_numpy(fashion_test)

print("Data prepared for SVM")


Data prepared for SVM


## Q1(b) – SVM Experiment 1  
Dataset: MNIST  
Kernel: poly


In [24]:
# SVM on MNIST with Polynomial Kernel
svm_poly = SVC(kernel='poly')

start_time = time.time()
svm_poly.fit(X_mnist_train, y_mnist_train)
end_time = time.time()

train_time_ms = (end_time - start_time) * 1000

y_pred = svm_poly.predict(X_mnist_test)
accuracy = accuracy_score(y_mnist_test, y_pred) * 100

print(f"MNIST + SVM (poly kernel)")
print(f"Test Accuracy: {accuracy:.2f}%")
print(f"Training Time: {train_time_ms:.2f} ms")


MNIST + SVM (poly kernel)
Test Accuracy: 98.09%
Training Time: 97204.21 ms


## Q1(b) – SVM Experiment 2  
Dataset: MNIST  
Kernel: rbf


In [25]:
# SVM on MNIST with RBF Kernel
svm_rbf = SVC(kernel='rbf')

start_time = time.time()
svm_rbf.fit(X_mnist_train, y_mnist_train)
end_time = time.time()

train_time_ms = (end_time - start_time) * 1000

y_pred = svm_rbf.predict(X_mnist_test)
accuracy = accuracy_score(y_mnist_test, y_pred) * 100

print(f"MNIST + SVM (rbf kernel)")
print(f"Test Accuracy: {accuracy:.2f}%")
print(f"Training Time: {train_time_ms:.2f} ms")


MNIST + SVM (rbf kernel)
Test Accuracy: 97.86%
Training Time: 122653.03 ms


## Q1(b) – SVM Experiment 3  
Dataset: FashionMNIST  
Kernel: poly


In [26]:
# SVM on FashionMNIST with Polynomial Kernel
svm_poly = SVC(kernel='poly')

start_time = time.time()
svm_poly.fit(X_fashion_train, y_fashion_train)
end_time = time.time()

train_time_ms = (end_time - start_time) * 1000

y_pred = svm_poly.predict(X_fashion_test)
accuracy = accuracy_score(y_fashion_test, y_pred) * 100

print(f"FashionMNIST + SVM (poly kernel)")
print(f"Test Accuracy: {accuracy:.2f}%")
print(f"Training Time: {train_time_ms:.2f} ms")


FashionMNIST + SVM (poly kernel)
Test Accuracy: 89.22%
Training Time: 139078.01 ms


## Q1(b) – SVM Experiment 4  
Dataset: FashionMNIST  
Kernel: rbf


In [27]:
# SVM on FashionMNIST with RBF Kernel
svm_rbf = SVC(kernel='rbf')

start_time = time.time()
svm_rbf.fit(X_fashion_train, y_fashion_train)
end_time = time.time()

train_time_ms = (end_time - start_time) * 1000

y_pred = svm_rbf.predict(X_fashion_test)
accuracy = accuracy_score(y_fashion_test, y_pred) * 100

print(f"FashionMNIST + SVM (rbf kernel)")
print(f"Test Accuracy: {accuracy:.2f}%")
print(f"Training Time: {train_time_ms:.2f} ms")


FashionMNIST + SVM (rbf kernel)
Test Accuracy: 88.60%
Training Time: 155279.37 ms


In [28]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


True
Tesla T4


In [29]:
from torch.utils.data import DataLoader

batch_size = 16

fashion_train_loader = DataLoader(fashion_train, batch_size=batch_size, shuffle=True, pin_memory=True)
fashion_val_loader   = DataLoader(fashion_val, batch_size=batch_size, shuffle=False, pin_memory=True)
fashion_test_loader  = DataLoader(fashion_test, batch_size=batch_size, shuffle=False, pin_memory=True)

print("FashionMNIST loaders ready for Q2")


FashionMNIST loaders ready for Q2


In [3]:
device = torch.device("cpu")
print("Using device:", device)


Using device: cpu


## Q2 Experiment 1  
Compute: CPU  
Dataset: FashionMNIST  
Model: ResNet-18  
Batch Size: 16  
Optimizer: SGD  
Learning Rate: 0.001  
Epochs: 5


In [17]:
import torch
import time

device = torch.device("cpu")

model = get_resnet("resnet18").to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

start_time = time.time()

train_acc, val_acc = train_model(
    model,
    fashion_train_loader,
    fashion_val_loader,
    optimizer,
    epochs=3,
    device=device
)

end_time = time.time()
train_time_ms = (end_time - start_time) * 1000

test_accuracy = test_model(model, fashion_test_loader, device)

print(f"CPU + ResNet-18 + SGD")
print(f"Train Time (ms): {train_time_ms:.2f}")
print(f"Test Accuracy (%): {test_accuracy:.2f}")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch [1/3] → Train Acc: 77.69% | Val Acc: 86.00%
Epoch [2/3] → Train Acc: 85.02% | Val Acc: 88.17%
Epoch [3/3] → Train Acc: 87.60% | Val Acc: 88.15%
Final Test Accuracy: 86.93%
CPU + ResNet-18 + SGD
Train Time (ms): 1851212.34
Test Accuracy (%): 86.93


## Q2 Experiment 2  
Compute: CPU  
Dataset: FashionMNIST  
Model: ResNet-18  
Batch Size: 16  
Optimizer: Adam  
Learning Rate: 0.001  
Epochs: 3


In [18]:
import torch
import time

device = torch.device("cpu")

model = get_resnet("resnet18").to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

start_time = time.time()

train_acc, val_acc = train_model(
    model,
    fashion_train_loader,
    fashion_val_loader,
    optimizer,
    epochs=3,
    device=device
)

end_time = time.time()
train_time_ms = (end_time - start_time) * 1000

test_accuracy = test_model(model, fashion_test_loader, device)

print("CPU + ResNet-18 + Adam")
print(f"Train Time (ms): {train_time_ms:.2f}")
print(f"Test Accuracy (%): {test_accuracy:.2f}")


Epoch [1/3] → Train Acc: 81.45% | Val Acc: 86.75%
Epoch [2/3] → Train Acc: 86.58% | Val Acc: 88.50%
Epoch [3/3] → Train Acc: 88.28% | Val Acc: 88.98%
Final Test Accuracy: 88.08%
CPU + ResNet-18 + Adam
Train Time (ms): 2748829.62
Test Accuracy (%): 88.08


## Q2 Experiment 3  
Compute: CPU  
Dataset: FashionMNIST  
Model: ResNet-50  
Batch Size: 16  
Optimizer: SGD  
Learning Rate: 0.001  
Epochs: 1


In [19]:
import torch
import time

device = torch.device("cpu")

model = get_resnet("resnet50").to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

start_time = time.time()

train_acc, val_acc = train_model(
    model,
    fashion_train_loader,
    fashion_val_loader,
    optimizer,
    epochs=1,   # ONLY 1 epoch
    device=device
)

end_time = time.time()
train_time_ms = (end_time - start_time) * 1000

test_accuracy = test_model(model, fashion_test_loader, device)

print("CPU + ResNet-50 + SGD")
print(f"Train Time (ms): {train_time_ms:.2f}")
print(f"Test Accuracy (%): {test_accuracy:.2f}")


Epoch [1/1] → Train Acc: 61.05% | Val Acc: 77.57%
Final Test Accuracy: 75.86%
CPU + ResNet-50 + SGD
Train Time (ms): 1395364.84
Test Accuracy (%): 75.86


## Q2 Experiment 4  
Compute: CPU  
Dataset: FashionMNIST  
Model: ResNet-50  
Batch Size: 16  
Optimizer: Adam  
Learning Rate: 0.001  
Epochs: 1


In [20]:
import torch
import time

device = torch.device("cpu")

model = get_resnet("resnet50").to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

start_time = time.time()

train_acc, val_acc = train_model(
    model,
    fashion_train_loader,
    fashion_val_loader,
    optimizer,
    epochs=1,   # ONLY 1 epoch
    device=device
)

end_time = time.time()
train_time_ms = (end_time - start_time) * 1000

test_accuracy = test_model(model, fashion_test_loader, device)

print("CPU + ResNet-50 + Adam")
print(f"Train Time (ms): {train_time_ms:.2f}")
print(f"Test Accuracy (%): {test_accuracy:.2f}")


Epoch [1/1] → Train Acc: 72.76% | Val Acc: 72.28%
Final Test Accuracy: 70.91%
CPU + ResNet-50 + Adam
Train Time (ms): 1878685.74
Test Accuracy (%): 70.91


In [15]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
Tesla T4


In [16]:
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler

def train_model(model, train_loader, val_loader, optimizer, epochs, device):
    criterion = nn.CrossEntropyLoss()
    model.to(device)

    scaler = GradScaler()  # AMP scaler

    train_acc_list = []
    val_acc_list = []

    for epoch in range(epochs):
        model.train()
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            with autocast():   # AMP forward pass
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc = 100 * correct / total
        train_acc_list.append(train_acc)

        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                with autocast():
                    outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = 100 * correct / total
        val_acc_list.append(val_acc)

        print(f"Epoch [{epoch+1}/{epochs}] → Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

    return train_acc_list, val_acc_list


In [17]:
device = torch.device("cuda")
print("Using device:", device)


Using device: cuda


In [18]:
from torch.utils.data import DataLoader

batch_size = 16

fashion_train_loader = DataLoader(fashion_train, batch_size=batch_size, shuffle=True, pin_memory=True)
fashion_val_loader   = DataLoader(fashion_val, batch_size=batch_size, shuffle=False, pin_memory=True)
fashion_test_loader  = DataLoader(fashion_test, batch_size=batch_size, shuffle=False, pin_memory=True)

print("FashionMNIST loaders ready (GPU)")


FashionMNIST loaders ready (GPU)


In [19]:
import time

model = get_resnet("resnet18").to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

start_time = time.time()

train_acc, val_acc = train_model(
    model,
    fashion_train_loader,
    fashion_val_loader,
    optimizer,
    epochs=5,
    device=device
)

end_time = time.time()
train_time_ms = (end_time - start_time) * 1000

test_accuracy = test_model(model, fashion_test_loader, device)

print("GPU + ResNet-18 + SGD")
print(f"Train Time (ms): {train_time_ms:.2f}")
print(f"Test Accuracy (%): {test_accuracy:.2f}")


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/tmp/ipython-input-3954169645.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()  # AMP scaler
/tmp/ipython-input-3954169645.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():   # AMP forward pass
/tmp/ipython-input-3954169645.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is 

Epoch [1/5] → Train Acc: 77.90% | Val Acc: 85.55%
Epoch [2/5] → Train Acc: 85.19% | Val Acc: 86.83%
Epoch [3/5] → Train Acc: 87.34% | Val Acc: 87.92%
Epoch [4/5] → Train Acc: 88.89% | Val Acc: 88.05%
Epoch [5/5] → Train Acc: 90.41% | Val Acc: 88.77%
Final Test Accuracy: 88.34%
GPU + ResNet-18 + SGD
Train Time (ms): 202796.74
Test Accuracy (%): 88.34


## Q2 Experiment (GPU)  
Model: ResNet-18  
Optimizer: Adam  
Batch Size: 16  
LR: 0.001  
Epochs: 5  
AMP: Enabled


In [20]:
import time

model = get_resnet("resnet18").to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

start_time = time.time()

train_acc, val_acc = train_model(
    model,
    fashion_train_loader,
    fashion_val_loader,
    optimizer,
    epochs=5,
    device=device
)

end_time = time.time()
train_time_ms = (end_time - start_time) * 1000

test_accuracy = test_model(model, fashion_test_loader, device)

print("GPU + ResNet-18 + Adam")
print(f"Train Time (ms): {train_time_ms:.2f}")
print(f"Test Accuracy (%): {test_accuracy:.2f}")


/tmp/ipython-input-3954169645.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()  # AMP scaler
/tmp/ipython-input-3954169645.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():   # AMP forward pass
/tmp/ipython-input-3954169645.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/5] → Train Acc: 81.15% | Val Acc: 86.00%
Epoch [2/5] → Train Acc: 86.66% | Val Acc: 87.20%
Epoch [3/5] → Train Acc: 88.40% | Val Acc: 88.00%
Epoch [4/5] → Train Acc: 89.62% | Val Acc: 89.08%
Epoch [5/5] → Train Acc: 90.57% | Val Acc: 90.32%
Final Test Accuracy: 90.34%
GPU + ResNet-18 + Adam
Train Time (ms): 244300.73
Test Accuracy (%): 90.34


 ## Q2 Experiment (GPU)  
Model: ResNet-50  
Optimizer: SGD  
Batch Size: 16  
LR: 0.001  
Epochs: 5  
AMP: Enabled


In [21]:
import time

model = get_resnet("resnet50").to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

start_time = time.time()

train_acc, val_acc = train_model(
    model,
    fashion_train_loader,
    fashion_val_loader,
    optimizer,
    epochs=5,
    device=device
)

end_time = time.time()
train_time_ms = (end_time - start_time) * 1000

test_accuracy = test_model(model, fashion_test_loader, device)

print("GPU + ResNet-50 + SGD")
print(f"Train Time (ms): {train_time_ms:.2f}")
print(f"Test Accuracy (%): {test_accuracy:.2f}")


/tmp/ipython-input-3954169645.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()  # AMP scaler
/tmp/ipython-input-3954169645.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():   # AMP forward pass
/tmp/ipython-input-3954169645.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/5] → Train Acc: 59.89% | Val Acc: 76.03%
Epoch [2/5] → Train Acc: 74.27% | Val Acc: 80.08%
Epoch [3/5] → Train Acc: 78.26% | Val Acc: 81.95%
Epoch [4/5] → Train Acc: 80.75% | Val Acc: 83.33%
Epoch [5/5] → Train Acc: 82.49% | Val Acc: 85.05%
Final Test Accuracy: 84.39%
GPU + ResNet-50 + SGD
Train Time (ms): 401587.10
Test Accuracy (%): 84.39


## Q2 Experiment (GPU)  
Model: ResNet-50  
Optimizer: Adam  
Batch Size: 16  
LR: 0.001  
Epochs: 5  
AMP: Enabled


In [22]:
import time

model = get_resnet("resnet50").to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

start_time = time.time()

train_acc, val_acc = train_model(
    model,
    fashion_train_loader,
    fashion_val_loader,
    optimizer,
    epochs=5,
    device=device
)

end_time = time.time()
train_time_ms = (end_time - start_time) * 1000

test_accuracy = test_model(model, fashion_test_loader, device)

print("GPU + ResNet-50 + Adam")
print(f"Train Time (ms): {train_time_ms:.2f}")
print(f"Test Accuracy (%): {test_accuracy:.2f}")


/tmp/ipython-input-3954169645.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()  # AMP scaler
/tmp/ipython-input-3954169645.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():   # AMP forward pass
/tmp/ipython-input-3954169645.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/5] → Train Acc: 71.50% | Val Acc: 81.40%
Epoch [2/5] → Train Acc: 80.21% | Val Acc: 83.55%
Epoch [3/5] → Train Acc: 81.72% | Val Acc: 84.67%
Epoch [4/5] → Train Acc: 85.11% | Val Acc: 86.62%
Epoch [5/5] → Train Acc: 85.79% | Val Acc: 82.45%
Final Test Accuracy: 81.62%
GPU + ResNet-50 + Adam
Train Time (ms): 499694.68
Test Accuracy (%): 81.62
